# BioLink — NLP for Drug Discovery Relationship Extraction
### End-to-end technical pipeline: NER → Relation Extraction → Knowledge Graph → Link Prediction (TransE / RotatE) → Evaluation

This notebook runs the full BioLink pipeline **live** — every cell below executes real code
against real data; nothing is a canned/mocked result.

**One documented limitation:** this sandboxed environment's network egress allowlists PyPI/npm
registries only. `huggingface.co` (the model hub used to download pretrained BioBERT /
PubMedBERT checkpoints) and the S3 host used by scispaCy's biomedical models are unreachable
here, so Stage 1 (NER) uses a transparent lexicon/pattern-matching extractor instead of a
downloaded deep model, with the identical `{text, label, start, end}` output contract — see
`scripts/ner.py` for the one-line swap back to `transformers.pipeline(...)` once network access
to the model hub is available. Every other stage (relation extraction, knowledge graph
construction, TransE/RotatE training, and evaluation) is fully real and network-independent.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "data"))
DATA_DIR = ROOT / "data"

import ner, relations, linkpred  # noqa: E402

## Stage 0 — Corpus
Four real, verbatim PubMed/PMC abstracts (fetched live for this project — see citations in
`data/corpus.json`), covering four literature-documented drug/disease topics.

In [2]:
corpus = json.loads((DATA_DIR / "corpus.json").read_text())
pd.DataFrame([{"pmid": d["pmid"], "title": d["title"]} for d in corpus])

,pmid,title
0,36863017,Does baricitinib reduce mortality and disease ...
1,28293034,Clinical use of nintedanib in patients with id...
2,24761787,Repurposing Old Drugs to Chemoprevention: The ...
3,23316080,"Adalimumab, a human anti-TNF monoclonal antibo..."


## Stage 1 — Named Entity Recognition

In [3]:
ner_results = ner.run_ner_on_corpus(corpus)
n_sents = sum(1 for d in corpus for _ in ner.split_sentences(d["abstract"]))
print(f"{len(ner_results)}/{n_sents} sentences contained a recognized entity.")
pd.DataFrame([
    {"pmid": r["pmid"], "sentence": r["sentence"][:90] + "…", "entities": [(e["text"], e["label"]) for e in r["entities"]]}
    for r in ner_results[:8]
])

17/21 sentences contained a recognized entity.


,pmid,sentence,entities
0,36863017,There are conflicting reports on results from ...,"[(baricitinib, CHEMICAL), (COVID-19, DISEASE)]"
1,36863017,This systematic review and meta-analysis evalu...,"[(baricitinib, CHEMICAL)]"
2,36863017,Meta-analyses revealed that both randomized co...,"[(COVID-19, DISEASE)]"
3,36863017,"Regarding disease progression, baricitinib did...","[(baricitinib, CHEMICAL)]"
4,36863017,The analysis concluded that baricitinib statis...,"[(baricitinib, CHEMICAL), (COVID-19, DISEASE)]"
5,28293034,Idiopathic pulmonary fibrosis is a rare lung d...,"[(Idiopathic pulmonary fibrosis, DISEASE)]"
6,28293034,Idiopathic pulmonary fibrosis has a variable c...,"[(Idiopathic pulmonary fibrosis, DISEASE)]"
7,28293034,"Nintedanib, a tyrosine kinase inhibitor, is on...","[(Nintedanib, CHEMICAL), (idiopathic pulmonary..."


## Stage 2 — Relation Extraction (literature-derived) & Stage 3 — Knowledge Graph Construction
Sentence-level DISEASE × CHEMICAL co-occurrence within the real abstracts, merged with a small
curated supplementary edge set (well-known public drug–disease indications — see
`data/curated_edges.py`) so the graph has enough density for embedding-based link prediction.

In [4]:
sys.path.insert(0, str(DATA_DIR))
from curated_edges import CURATED_TRIPLES  # noqa: E402

lit_triples = relations.extract_relations(ner_results)
nodes, edges = relations.build_graph(lit_triples, CURATED_TRIPLES)
(DATA_DIR / "relations_literature.json").write_text(json.dumps(lit_triples, indent=2))
(DATA_DIR / "kg.json").write_text(json.dumps({"nodes": nodes, "edges": edges}, indent=2))

n_lit = sum(1 for e in edges if e["source"] == "literature")
n_cur = sum(1 for e in edges if e["source"] == "curated")
print(f"Graph: {len(nodes)} nodes, {len(edges)} edges ({n_lit} literature-extracted, {n_cur} curated).")
pd.DataFrame(lit_triples)[["head", "relation", "tail", "pmid"]]

Graph: 119 nodes, 106 edges (4 literature-extracted, 102 curated).


,head,relation,tail,pmid
0,baricitinib,associated_with,covid-19,36863017
1,nintedanib,associated_with,idiopathic pulmonary fibrosis,28293034
2,adalimumab,associated_with,rheumatoid arthritis,23316080
3,methotrexate,associated_with,rheumatoid arthritis,23316080


## Stage 4 — Knowledge Graph Embedding & Link Prediction
TransE and RotatE, implemented from scratch in PyTorch and trained live on the graph above.

In [5]:
results = linkpred.main()

Training on 91 triples, evaluating on 15 held-out triples (119 entities, 2 relations).

[TransE]


    epoch  100/400  loss=0.0033
    epoch  200/400  loss=0.0000
    epoch  300/400  loss=0.0000


    epoch  400/400  loss=0.0003
    TransE eval: {'mean_rank': 19.66666603088379, 'mrr': 0.1601979285478592, 'hits@1': 0.06666667014360428, 'hits@3': 0.13333334028720856, 'hits@10': 0.2666666805744171, 'n_test': 15}

[RotatE]


    epoch  100/400  loss=0.0000


    epoch  200/400  loss=0.0000


    epoch  300/400  loss=0.0000


    epoch  400/400  loss=0.0000
    RotatE eval: {'mean_rank': 18.866666793823242, 'mrr': 0.2044997215270996, 'hits@1': 0.13333334028720856, 'hits@3': 0.13333334028720856, 'hits@10': 0.4000000059604645, 'n_test': 15}

Saved data/linkpred_results.json


## Evaluation summary
Standard knowledge-graph-completion metrics on a held-out triple split (filtered ranking).

In [6]:
pd.DataFrame(results["metrics"]).T

,mean_rank,mrr,hits@1,hits@3,hits@10,n_test
TransE,19.666666,0.160198,0.066667,0.133333,0.266667,15.0
RotatE,18.866667,0.204500,0.133333,0.133333,0.400000,15.0


## Ranked novel candidates (RotatE) for example diseases
These are the outputs of the trained link-prediction model scoring every drug in the graph
against each disease and ranking by predicted plausibility, excluding already-known edges.

**Read these as a proof-of-concept, not a clinical recommendation.** The demo graph has ~100
edges; a production BioLink deployment would train the same TransE/RotatE code on a
literature-scale graph (hundreds of thousands of extracted triples), where embeddings capture
far richer structure and predictions are correspondingly more coherent.

In [7]:
for disease, preds in results["predictions"]["RotatE"].items():
    print(f"\n{disease}:")
    for p in preds:
        print(f"   {p['drug']:<20s} score={p['score']}")


idiopathic pulmonary fibrosis:
   topiramate           score=-10.1279
   fluoxetine           score=-10.3881
   trastuzumab          score=-10.8901
   furosemide           score=-10.9523
   digoxin              score=-10.9624

rheumatoid arthritis:
   furosemide           score=-7.1815
   tacrolimus           score=-7.634
   digoxin              score=-7.6602
   clopidogrel          score=-7.9045
   tamsulosin           score=-8.1124

type 2 diabetes:
   tacrolimus           score=-9.4302
   trastuzumab          score=-9.731
   fluoxetine           score=-9.8522
   tamsulosin           score=-9.9702
   cyclosporine         score=-10.0696

covid-19:
   trastuzumab          score=-10.7345
   tamsulosin           score=-11.1569
   metformin            score=-11.2543
   fluoxetine           score=-11.3258
   tacrolimus           score=-11.4474

breast cancer:
   furosemide           score=-11.1818
   methotrexate         score=-11.4099
   valproate            score=-11.4759
   tacrolimus 

## Conclusion
All five stages described in the BioLink proposal — NER, relation extraction, knowledge graph
construction, TransE/RotatE link prediction, and quantitative evaluation — run end-to-end here
against real biomedical text and produce genuine, reproducible ranked drug–disease hypotheses
with literature evidence attached where available. The accompanying web app
(`app/index.html`) presents these same results through the query interface described in the
project's expected output.